In [ ]:
import geopandas as gpd
import rasterio
import shapely

from geofeatureviz import map_style, preprocessor, topography
from geofeatureviz.io import loader, path_settings
from geofeatureviz.projections import Equirectangular
from geofeatureviz.svg_handler import MapSVG

RESULT_DIRECTORY = path_settings.results_dir / "german_mountains"
SIMPLIFICATION = 0.015

BBOX = shapely.geometry.box(5.5, 47.2, 15.5, 55.1)  # bounds by Wikipedia (in lon/lat)

CANVAS_KWARGS = {
    "bounds": BBOX.bounds,
    "width": 500,
    "projection": Equirectangular(y_scale=1.5),  # projection used by Wikipedia
}

# Load Countries and States

In [ ]:
# countries
countries = preprocessor.prep_dataset(feature="country", source="ne", resolution=10)
countries.geometry = countries.geometry.simplify(SIMPLIFICATION, preserve_topology=True)
german_vicinity = countries.clip(BBOX)
germany = german_vicinity[german_vicinity["admin"] == "Germany"]
german_vicinity = german_vicinity[~(german_vicinity["admin"] == "Germany")]

# states
all_states = preprocessor.prep_dataset(feature="state", source="ne", resolution=10)
states = all_states[all_states["admin"] == "Germany"].copy()
states.geometry = states.geometry.simplify(SIMPLIFICATION, preserve_topology=True)

# Load Mountain Data
This includes the topographic elevation data and the geometries of the mountain ranges that I want to mark.

In [ ]:
data_file = path_settings.data_dir / "COPERNICUS_90_DEM.tiff"

with rasterio.open(data_file) as raster_src:
    thresholds = topography.get_thresholds(raster_src.read(1), n_thresholds=15)
    elevation_gdf = topography.raster_to_elevation_gdf(raster_src, thresholds)

cmap = map_style.get_elevation_cmap()
norm = map_style.ElevationNorm(list(thresholds))
colors = cmap.get_hex(norm(elevation_gdf["elevation"]))
elevation_gdf["color"] = colors

elevation_gdf = gpd.overlay(elevation_gdf, countries, how="intersection")
elevation_gdf = elevation_gdf.clip(BBOX).sort_index()

mountain_gdf = loader.load_dataset("mountain_germany", source="osm", resolution=10)
mountain_gdf.geometry = mountain_gdf.geometry.simplify(SIMPLIFICATION)

# Topographic Map
Create a topographic map of Germany and save it as SVG.

In [ ]:
canvas = MapSVG(**CANVAS_KWARGS)
canvas.add_background(map_style.COLORS["lake"])

for row in elevation_gdf.itertuples():
    geom_id = f"elevation_{int(round(row.elevation))}"
    svg_elem = canvas.geom_to_svg(row.geometry, geometry_id=geom_id, fill=row.color)
    canvas.add(svg_elem)

# styles
land_style = dict(map_style.STYLES["land"])
land_style["fill"] = "none"
land_style["stroke_opacity"] = 0.33
other_land_style = dict(land_style)
other_land_style["fill"] = "#808080"
other_land_style["fill_opacity"] = 0.5
state_style = dict(land_style)
state_style["stroke_width"] = 0.33

# countries and states
canvas.add_gdf(germany, "Germany", **land_style)
canvas.add_gdf(states, "States", **state_style)
canvas.add_gdf(german_vicinity, "Countries", **other_land_style)

# save the original and and optimized version
canvas.save(RESULT_DIRECTORY / "Germany_mountain_map.svg")
canvas.save(RESULT_DIRECTORY / "optimized" / "Germany_mountain_map.svg", optimize=True)

canvas

# Mark Mountain Ranges
First, I want to show all mountain ranges on a single canvas, which I don't need to save.

In [ ]:
mountain_canvas = canvas.copy()
mountain_marker_style = map_style.STYLES["neutral_marker"]
mountain_canvas.add_gdf(mountain_gdf, "MountainRanges", **mountain_marker_style)
mountain_canvas

# Mark All Mountain Ranges Independently
For each mountain range, a canvas of the same size as before is drawn, with a single mountain range marked on it. This can be used to lay the SVG-file saved here on top of the SVG-file of the Germany mountain map, making it look like the mountain range is marked on the Germany mountain map.

In [ ]:
mountain_marker_style = dict(map_style.STYLES["highlight_marker"])
mountain_marker_style["fill_opacity"] = 0.2

for row in mountain_gdf.itertuples():
    mountain_name = row.name
    geom = row.geometry

    canvas = MapSVG(**CANVAS_KWARGS)
    svg_elem = canvas.geom_to_svg(
        geom, geometry_id=mountain_name, **mountain_marker_style
    )
    canvas.add(svg_elem)

    canvas.save(RESULT_DIRECTORY / f"Germany_mountain_map_{mountain_name}.svg")
    canvas.save(
        RESULT_DIRECTORY / "optimized" / f"Germany_mountain_map_{mountain_name}.svg",
        optimize=True,
    )